In [15]:
import os
import numpy as np
import pandas as pd
import nibabel as nib

from scipy import ndimage
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityd,
    CropForegroundd,
    ResizeD,
    EnsureTyped
)
from monai.data import Dataset

In [16]:
#loading the trained model from segmentation(U-Net)
import torch
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,   # <-- change this
    channels=(16,32,64,128,256),
    strides=(2,2,2,2),
    num_res_units=2
).to(device)

model.load_state_dict(torch.load("../models/pancreas_unet.pth", map_location=device))
model.eval()

print("Model Loaded")

Model Loaded


In [17]:
#extract features from the segmented pancreas( volume, mean intensity, std intensity, max intensity, min intensity)
def extract_features(image, mask):

    pancreas_pixels = image[mask > 0]

    if len(pancreas_pixels) == 0:
        return None

    features = {
        "volume": np.sum(mask),
        "mean_intensity": np.mean(pancreas_pixels),
        "std_intensity": np.std(pancreas_pixels),
        "max_intensity": np.max(pancreas_pixels),
        "min_intensity": np.min(pancreas_pixels),
    }

    return features

In [18]:
import pandas as pd
import os

val_df = pd.read_csv("../csv_lists/val.csv")

In [19]:
val_files = [
    {
        "image": os.path.normpath(os.path.join("..", img)),
        "label": os.path.normpath(os.path.join("..", lbl)),
    }
    for img, lbl in zip(val_df["image"], val_df["label"])
]

In [20]:
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(
        keys=["image", "label"],
        axcodes="RAS"
    ),
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5,1.5,2.0),
        mode=("bilinear","nearest")
    ),
    ScaleIntensityd(keys=["image"]),
    CropForegroundd(
        keys=["image","label"],
        source_key="image"
    ),
    ResizeD(
        keys=["image","label"],
        spatial_size=(128,128,128)
    ),
    EnsureTyped(keys=["image","label"])
])

c:\Pancreas_\.venv\Lib\site-packages\monai\utils\deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [21]:
from monai.data import Dataset

val_ds = Dataset(
    data=val_files,
    transform=val_transforms
)

print("Validation Dataset Created")

Validation Dataset Created


In [22]:
#testing a sample image and mask

sample = val_ds[0]

image = sample["image"][0].numpy()
mask = sample["label"][0].numpy()

features = extract_features(image, mask)

print(features)

{'volume': np.float32(5531.37), 'mean_intensity': np.float32(0.4608415), 'std_intensity': np.float32(0.018506464), 'max_intensity': np.float32(0.5059156), 'min_intensity': np.float32(0.23280771)}


In [23]:
all_features = []

for sample in val_ds:

    image = sample["image"][0].numpy()
    mask = sample["label"][0].numpy()

    features = extract_features(image, mask)

    if features is not None:
        all_features.append(features)

print("Total samples:", len(all_features))

Total samples: 57


In [24]:
features_df = pd.DataFrame(all_features)

features_df.head()

,volume,mean_intensity,std_intensity,max_intensity,min_intensity
0,5531.370117,0.460842,0.018506,0.505916,0.232808
1,4201.597168,0.419074,0.017558,0.646429,0.298106
2,1984.020874,0.394272,0.021166,0.439039,0.296707
3,2822.166504,0.365470,0.014340,0.408287,0.307319
4,3140.381836,0.316132,0.014014,0.361658,0.256216


In [25]:
#saving features
import os

os.makedirs("../features", exist_ok=True)

features_df.to_csv("../features/features.csv", index=False)

print("Features saved successfully!")

Features saved successfully!
